# 124 — Supervisor-workers

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** (a) promedio = 2.3/3 ≈ **0.7667**. (b) mínimo = security 0.6 < 0.7 →
**"mejorar security"** (la del laboratorio). (c) ponderada = 0.3·0.8 + 0.5·0.6 +
0.2·0.9 = 0.24 + 0.30 + 0.18 = **0.72** — más baja que el promedio porque el peso
carga el aspecto débil. (d) veto: 0.6 < 0.65 → **bloqueado**, sin importar el resto.
Cambia la decisión la política (c) solo si se usara como decisión con umbral 0.75
(0.72 < 0.75 rechaza); (b) y (d) coinciden en señalar seguridad.

**Ejercicio 2.** Ejemplo válido:

```text
Objetivo: evaluar la postura de seguridad del repo `demo` para publicación.
Revisa: presencia de threat model, secretos en el código, dependencias con CVE.
Salida EXACTA: {"agent": "security", "score": 0..1, "finding": "<un hallazgo>"}
Herramientas: lectura del repo y SBOM; prohibido modificar archivos.
Límite: una pasada, máx. 10 archivos; si no concluyes, score null + motivo.
```

**Ejercicio 3.** Clave: `None` se excluye del cálculo y se registra en `limitations`;
un caído jamás entra como 0.

**Ejercicio 4.** Se localiza el worker de score mínimo y se comprueba que su nombre
(mapeado a español, "seguridad") aparece en la decisión.


In [ ]:
result = run_lab("multiagent", seed=124)
assert result["kind"] == "multiagent"
assert result["evidence"]
show(result)


In [ ]:
# Ejercicio 1
scores = {"quality": 0.8, "security": 0.6, "documentation": 0.9}
pesos = {"quality": 0.3, "security": 0.5, "documentation": 0.2}
promedio = round(sum(scores.values()) / 3, 4)
peor = min(scores, key=scores.get)
decision_min = f"mejorar {peor}" if scores[peor] < 0.7 else "aprobar"
ponderada = round(sum(scores[k] * pesos[k] for k in scores), 4)
decision_veto = "bloqueado por security" if scores["security"] < 0.65 else "pasa veto"
print(promedio, decision_min, ponderada, decision_veto)

# Ejercicio 3
def consolidar(workers, umbral=0.7):
    vivos = [w for w in workers if w["score"] is not None]
    caidos = [w["agent"] for w in workers if w["score"] is None]
    overall = round(sum(w["score"] for w in vivos) / len(vivos), 4)
    peor = min(vivos, key=lambda w: w["score"])
    decision = (f"mejorar {peor['agent']}" if peor["score"] < umbral else "aprobar")
    return {"overall": overall, "decision": decision,
            "limitations": [f"worker {a} sin respuesta: dato ausente, no 0" for a in caidos]}

demo = [{"agent": "quality", "score": 0.8, "finding": "tests"},
        {"agent": "security", "score": None, "finding": "timeout"},
        {"agent": "documentation", "score": 0.9, "finding": "guías"}]
show(consolidar(demo))

# Ejercicio 4
result = run_lab("multiagent", seed=124)
workers = result["result"]["workers"]
peor_worker = min(workers, key=lambda w: w["score"])
nombres = {"security": "seguridad", "quality": "calidad", "documentation": "documentación"}
assert nombres[peor_worker["agent"]] in result["result"]["supervisor"]["decision"]
print("la decisión sigue al mínimo:", peor_worker["agent"],
      "→", result["result"]["supervisor"]["decision"])


## Reflexión

1. Si `security` devolviera timeout en vez de 0.6, ¿qué política aplicarías (reintento, degradación, aborto) y por qué el promedio con los 2 restantes (0.85) sería una conclusión engañosa?
2. El supervisor de este laboratorio es una función determinista. Con un supervisor LLM, ¿qué parte del ciclo (descomponer, asignar, consolidar) esperas que falle primero y qué evidencia recogerías para demostrarlo?
3. ¿Cuándo preferirías veto puro de seguridad ("cualquier score < 0.7 bloquea") frente a la regla del mínimo, y qué coste operativo tiene un veto con falsos positivos?
